In [2]:
import pandas as pd 


df = pd.read_csv('data/works-2025-05-01T01-14-16.csv')

/var/folders/6z/w7qyc8h96b92v7zs458_gfj40000gp/T/ipykernel_9007/944277725.py:4: DtypeWarning: Columns (15,22,34,113,175,176,177,178,179,180,181,182) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/works-2025-05-01T01-14-16.csv')


In [6]:
df.shape

(18697, 183)

In [ ]:
import requests
import json
from queue import Queue
import threading
import time
import os

JSON_PATH = "data/openalex_data.json"
MAX_THREADS = 5  
REQUEST_RATE = 1 
SAVE_POINTS = [0.5, 1.0]  

os.makedirs('data',exist_ok=True)
try:
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        existing_data = json.load(f)
    existing_ids = {work["id"].split("/")[-1] for work in existing_data}
    print(f"🔍 {len(existing_data)} registros cargados previamente")
except FileNotFoundError:
    existing_data = []
    existing_ids = set()
    print("🆕 Creando nuevo archivo JSON")

# Variables compartidas
json_lock = threading.Lock()
progress_lock = threading.Lock()
total_processed = 0
total_to_process = 0
new_data_buffer = [] 
last_save_point = 0  


def save_data():
    global existing_data, new_data_buffer
    with json_lock:
        existing_data.extend(new_data_buffer)
        with open(JSON_PATH, "w", encoding="utf-8") as f:
            json.dump(existing_data, f, indent=4, ensure_ascii=False)
        new_data_buffer = []
        print(f"💾 Datos guardados. Total actual: {len(existing_data)} registros")


def check_save_point(progress):
    global last_save_point
    for point in SAVE_POINTS:
        if last_save_point < point <= progress:
            save_data()
            last_save_point = point
            return True
    return False


def worker(queue):
    global total_processed, new_data_buffer

    while not queue.empty():
        start_time = time.time()
        id = queue.get()

        url = f"https://api.openalex.org/works/{id}"
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            work_data = response.json()

            with json_lock:
                new_data_buffer.append(work_data)
                existing_ids.add(id)

            with progress_lock:
                total_processed += 1
                progress = total_processed / total_to_process
                elapsed = time.time() - start_time
                print(
                    f"✅ [{threading.current_thread().name}] {id} procesado | "
                    f"Progreso: {total_processed}/{total_to_process} "
                    f"({progress*100:.1f}%) | "
                    f"Tiempo: {elapsed:.2f}s"
                )

                check_save_point(progress)

        except requests.exceptions.RequestException as e:
            with progress_lock:
                print(
                    f"❌ [{threading.current_thread().name}] Error con {id}: {str(e)}"
                )
                queue.put(id)  
        except Exception as e:
            with progress_lock:
                print(
                    f"⚠️ [{threading.current_thread().name}] Error inesperado con {id}: {str(e)}"
                )

        processing_time = time.time() - start_time
        sleep_time = max(0, 1.0 - processing_time)
        time.sleep(sleep_time)

        queue.task_done()


queue = Queue()
for _, row in df.iterrows():
    id = row["id"].split("/")[-1]
    if id not in existing_ids:
        queue.put(id)

total_to_process = queue.qsize()
print(f"🚀 Iniciando descarga de {total_to_process} nuevos registros")


threads = []
for i in range(min(MAX_THREADS, queue.qsize())):
    t = threading.Thread(target=worker, args=(queue,), name=f"Worker-{i+1}")
    t.daemon = True
    t.start()
    threads.append(t)


try:
    while any(t.is_alive() for t in threads):
        time.sleep(5)
        with progress_lock:
            remaining = queue.qsize()
            processed = total_processed
            progress = processed / total_to_process
            print(
                f"\n📊 Progreso actual: {processed}/{total_to_process} "
                f"({progress*100:.1f}%) | "
                f"Restantes: {remaining} | "
                f"Hilos activos: {sum(1 for t in threads if t.is_alive())}\n"
            )
            check_save_point(progress)

except KeyboardInterrupt:
    print("\n🛑 Interrupción recibida, guardando datos recolectados...")
    save_data()


queue.join()
save_data()  
print(f"\n🎉 ¡Proceso completado! JSON actualizado en '{JSON_PATH}'")
print(f"📂 Total de registros: {len(existing_data)}")

🔍 18697 registros cargados previamente
🚀 Iniciando descarga de 0 nuevos registros
💾 Datos guardados. Total actual: 18697 registros

🎉 ¡Proceso completado! JSON actualizado en 'openalex_data.json'
📂 Total de registros: 18697


In [ ]:
import networkx as nx
import json
from tqdm import tqdm
from pyvis.network import Network


with open("data/openalex_data.json", "r") as f:
    data = json.load(f)


G = nx.Graph()

i = 0
for work in tqdm(data, desc="Procesando publicaciones"):

    G.add_node(
        work["id"].split("/")[-1] if work["id"].split("/")[-1] else i,
        label=work["title"] if work["title"] else work["display_name"],
        type="work",
        year=work["publication_year"] if work["publication_year"] else "unknown",
    )

    for authorship in work["authorships"]:
        author_id = authorship["author"]["id"]
        author_name = authorship["author"]["display_name"]
        if not author_id or not author_name:
            continue
        if not G.has_node(author_id):
            G.add_node(
                author_id,
                label=author_name if author_name else "",
                type="author",
                orcid=(
                    authorship["author"]["orcid"].split("/")[-1]
                    if authorship["author"]["orcid"]
                    else ""
                ),
            )

        G.add_edge(author_id, work["id"].split('/')[-1]  if work["id"].split("/")[-1] else i)
    i += 1


print('nodos:',G.number_of_nodes())
print('aristas:',G.number_of_edges())
net = Network(height="800px", notebook=False)
net.from_nx(G)  
net.show("grafo.html")

Procesando publicaciones: 100%|██████████| 18697/18697 [00:00<00:00, 44688.93it/s]


nodos: 48897
aristas: 71430
